In [4]:
import os
import re
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

In [107]:
def read_file(file_path):
    with open(file_path, 'r', encoding="utf-8") as file:
        return file.read()

def extract_operators(text):
    operators = re.findall(r'\[(AND|OR|NOT)\]', text)
    return operators

def count_operators(operators):
    return {
        'AND': operators.count('AND'),
        'OR': operators.count('OR'),
        'NOT': operators.count('NOT')
    }

def compare_operators(label_text, model_text):
    label_operators = extract_operators(label_text)
    model_operators = extract_operators(model_text)

    label_counts = count_operators(label_operators)
    model_counts = count_operators(model_operators)

    relaxed_comparison = {
        'AND': label_counts['AND'] == model_counts['AND'],
        'OR': label_counts['OR'] == model_counts['OR'],
        'NOT': label_counts['NOT'] == model_counts['NOT']
    }

    def exact_position_comparison(label_text, model_text, operator):
        pattern = re.compile(rf'(\w+)\s+\[{operator}\]')
        label_positions = [(match.group(1), match.start()) for match in pattern.finditer(label_text)]
        model_positions = [(match.group(1), match.start()) for match in pattern.finditer(model_text)]


        return label_positions == model_positions

    exact_comparison = {
        'AND': exact_position_comparison(label_text, model_text, 'AND'),
        'OR': exact_position_comparison(label_text, model_text, 'OR'),
        'NOT': exact_position_comparison(label_text, model_text, 'NOT')
    }

    return relaxed_comparison, exact_comparison, label_counts, model_counts



def extract_nct_number(filename):
    match = re.search(r'NCT\d+', filename)
    return match.group() if match else None

def process_files(label_folder, model_folder):
    relaxed_comparisons = []
    exact_comparisons = []
    label_operator_counts = []
    model_operator_counts = []

    for model_file in os.listdir(model_folder):
        if model_file.endswith('.txt'):
            nct_number = extract_nct_number(model_file)

            if nct_number:
                label_file_path = os.path.join(label_folder, f'{nct_number}.txt')
                model_file_path = os.path.join(model_folder, model_file)

                if os.path.exists(label_file_path):
                    label_text = read_file(label_file_path)
                    model_text = read_file(model_file_path)

                    relaxed_comparison, exact_comparison, label_counts, model_counts = compare_operators(label_text, model_text)
                    print(label_counts)
                    print(model_counts)
                    
                    relaxed_comparisons.append(relaxed_comparison)
                    exact_comparisons.append(exact_comparison)
                    label_operator_counts.append(label_counts)
                    model_operator_counts.append(model_counts)

    return relaxed_comparisons, exact_comparisons, label_operator_counts, model_operator_counts




In [108]:
label_folder = '../../input/lct_p1'
model_folder = 'model_output/Llama-3-70B-Instruct_4_shot/output'

relaxed_comparisons, exact_comparisons, label_operator_counts, model_operator_counts = process_files(label_folder, model_folder)

{'AND': 0, 'OR': 1, 'NOT': 0}
{'AND': 1, 'OR': 1, 'NOT': 0}
{'AND': 0, 'OR': 0, 'NOT': 1}
{'AND': 1, 'OR': 0, 'NOT': 0}
{'AND': 0, 'OR': 3, 'NOT': 0}
{'AND': 1, 'OR': 4, 'NOT': 1}
{'AND': 2, 'OR': 1, 'NOT': 0}
{'AND': 2, 'OR': 1, 'NOT': 0}
{'AND': 0, 'OR': 1, 'NOT': 0}
{'AND': 0, 'OR': 1, 'NOT': 0}
{'AND': 0, 'OR': 2, 'NOT': 0}
{'AND': 2, 'OR': 0, 'NOT': 0}
{'AND': 0, 'OR': 1, 'NOT': 1}
{'AND': 1, 'OR': 1, 'NOT': 0}
{'AND': 5, 'OR': 2, 'NOT': 0}
{'AND': 2, 'OR': 1, 'NOT': 0}
{'AND': 1, 'OR': 2, 'NOT': 2}
{'AND': 2, 'OR': 0, 'NOT': 2}
{'AND': 0, 'OR': 10, 'NOT': 0}
{'AND': 2, 'OR': 7, 'NOT': 0}
{'AND': 0, 'OR': 1, 'NOT': 0}
{'AND': 0, 'OR': 1, 'NOT': 1}
{'AND': 1, 'OR': 11, 'NOT': 0}
{'AND': 5, 'OR': 4, 'NOT': 1}
{'AND': 1, 'OR': 7, 'NOT': 5}
{'AND': 0, 'OR': 5, 'NOT': 2}
{'AND': 0, 'OR': 1, 'NOT': 0}
{'AND': 0, 'OR': 1, 'NOT': 0}
{'AND': 0, 'OR': 0, 'NOT': 0}
{'AND': 1, 'OR': 0, 'NOT': 0}
{'AND': 0, 'OR': 5, 'NOT': 2}
{'AND': 1, 'OR': 3, 'NOT': 0}
{'AND': 1, 'OR': 2, 'NOT': 2}
{'AND': 

In [104]:
total_and = sum(count['AND'] for count in label_operator_counts)
total_or = sum(count['OR'] for count in label_operator_counts)
total_not = sum(count['NOT'] for count in label_operator_counts)

print("Total AND:", total_and)
print("Total OR:", total_or)
print("Total NOT:", total_not)

Total AND: 819
Total OR: 4153
Total NOT: 948


In [105]:
total_and = sum(count['AND'] for count in model_operator_counts)
total_or = sum(count['OR'] for count in model_operator_counts)
total_not = sum(count['NOT'] for count in model_operator_counts)

print("Total AND:", total_and)
print("Total OR:", total_or)
print("Total NOT:", total_not)

Total AND: 1458
Total OR: 3383
Total NOT: 445


In [98]:
import os
import re
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

def read_file(file_path):
    with open(file_path, 'r', encoding="utf-8") as file:
        return file.read()

def find_operator_words(text, operator):
    if operator == 'NOT':
        pattern = re.compile(rf'\[{operator}\]\s+(\w+)?')
    else:
        pattern = re.compile(rf'(\w+)?\s*\[{operator}\]')

    matches = pattern.findall(text)
    words = [match for match in matches if match]
    return words

def compare_operators(label_text, model_text, operator):
    label_words = find_operator_words(label_text, operator)
    model_words = find_operator_words(model_text, operator)

    tp = sum(1 for word in label_words if word in model_words)
    fp = sum(1 for word in model_words if word not in label_words)
    fn = sum(1 for word in label_words if word not in model_words)

    return tp, fp, fn, len(label_words), len(model_words)

def calculate_metrics(tp, fp, fn):
    precision = tp / (tp + fp) if tp + fp > 0 else 0
    recall = tp / (tp + fn) if tp + fn > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if precision + recall > 0 else 0
    accuracy = tp / (tp + fn) if tp + fn > 0 else 0
    return precision, recall, f1, accuracy

def evaluate_models(label_folder, model_folder):
    operators = ['AND', 'OR', 'NOT']
    metrics = {op: {'tp': 0, 'fp': 0, 'fn': 0, 'label_count': 0, 'model_count': 0, 'correct_count': 0, 'correct_ncts': []} for op in operators}
    processed_label_files = 0

    for model_file in os.listdir(model_folder):
        if model_file.endswith('.txt'):
            nct_number = extract_nct_number(model_file)

            if nct_number:
                label_file_path = os.path.join(label_folder, f'{nct_number}.txt')
                model_file_path = os.path.join(model_folder, model_file)

                if os.path.exists(label_file_path):
                    processed_label_files += 1
                    label_text = read_file(label_file_path)
                    model_text = read_file(model_file_path)

                    for op in operators:
                        tp, fp, fn, label_count, model_count = compare_operators(label_text, model_text, op)

                        metrics[op]['tp'] += tp
                        metrics[op]['fp'] += fp
                        metrics[op]['fn'] += fn
                        metrics[op]['label_count'] += label_count
                        metrics[op]['model_count'] += model_count
                        metrics[op]['correct_count'] += tp

                        if tp > 0:
                            metrics[op]['correct_ncts'].append(nct_number)

    print(f"Total processed label files: {processed_label_files}")
    for op in operators:
        precision, recall, f1, accuracy = calculate_metrics(metrics[op]['tp'], metrics[op]['fp'], metrics[op]['fn'])
        print(f"{op}:")
        print(f"  Precision: {precision:.3f}")
        print(f"  Recall: {recall:.3f}")
        print(f"  F1-score: {f1:.3f}")
        print(f"  Accuracy: {accuracy:.3f}")
        print(f"  Total in Label: {metrics[op]['label_count']}")
        print(f"  Total in Model: {metrics[op]['model_count']}")
        print(f"  Correctly Identified: {metrics[op]['correct_count']}")
        #print(f"  Correct NCTs: {metrics[op]['correct_ncts']}\n")

def extract_nct_number(filename):
    match = re.search(r'NCT\d+', filename)
    return match.group() if match else None

label_folder = '../../input/lct_p1'
model_folder = 'model_output/Llama-3-70B-Instruct_4_shot/output'

evaluate_models(label_folder, model_folder)


Total processed label files: 1006
AND:
  Precision: 0.155
  Recall: 0.259
  F1-score: 0.194
  Accuracy: 0.259
  Total in Label: 740
  Total in Model: 1242
  Correctly Identified: 192
OR:
  Precision: 0.597
  Recall: 0.686
  F1-score: 0.638
  Accuracy: 0.686
  Total in Label: 2451
  Total in Model: 2804
  Correctly Identified: 1681
NOT:
  Precision: 0.031
  Recall: 0.014
  F1-score: 0.019
  Accuracy: 0.014
  Total in Label: 948
  Total in Model: 419
  Correctly Identified: 13


In [31]:
def read_file(file_path):
    with open(file_path, 'r', encoding="utf-8") as file:
        return file.read()

def extract_operators(text):
    operator_pattern = r'\b(\w+)?\s*[\.\,\!\?\;\:]*\s*\[(AND|OR|NOT)\]\s*(\w+)?\b'
    matches = re.findall(operator_pattern, text)
    operators = []

    for before, operator, after in matches:
        if operator == 'AND' or operator == 'OR':
            if before and after:
                operators.append((operator, before, after))
        elif operator == 'NOT':
            if after:
                operators.append((operator, after))

    return operators

def extract_nct_number(filename):
    match = re.search(r'NCT\d+', filename)
    return match.group() if match else None

def process_files(label_folder, model_folder):
    for model_file in os.listdir(model_folder):
        if model_file.endswith('.txt'):
            nct_number = extract_nct_number(model_file)
            if nct_number:
                label_file_path = os.path.join(label_folder, f'{nct_number}.txt')
                model_file_path = os.path.join(model_folder, model_file)
                if os.path.exists(label_file_path):
                    label_text = read_file(label_file_path)
                    model_text = read_file(model_file_path)
                    label_operators = extract_operators(label_text)
                    model_operators = extract_operators(model_text)
                    print(label_operators)
                    print(model_operators)

label_folder = '../../input/lct_p1'
model_folder = 'model_output/Llama-3-70B-Instruct_4_shot/output'
process_files(label_folder, model_folder)

OR:
  Precision: 0.234
  Recall: 0.247
  F1-score: 0.240
  Accuracy: 0.137
  Total in Label: 2315
  Total in Model: 2443
  Correctly Identified: 572

AND:
  Precision: 0.056
  Recall: 0.119
  F1-score: 0.076
  Accuracy: 0.040
  Total in Label: 511
  Total in Model: 1084
  Correctly Identified: 61

NOT:
  Precision: 0.058
  Recall: 0.115
  F1-score: 0.077
  Accuracy: 0.040
  Total in Label: 96
  Total in Model: 190
  Correctly Identified: 11



In [40]:
import os
import re
from collections import defaultdict

def read_file(file_path):
    with open(file_path, 'r', encoding="utf-8") as file:
        return file.read()

def extract_operators(text):
    operators = re.findall(r'\[(AND|OR|NOT)\]', text)
    return operators

def count_operators(operators):
    return {
        'AND': operators.count('AND'),
        'OR': operators.count('OR'),
        'NOT': operators.count('NOT')
    }

def extract_nct_number(filename):
    match = re.search(r'NCT\d+', filename)
    return match.group() if match else None

def calculate_metrics(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0

    return precision, recall, f1, accuracy

def process_files(label_folder, model_folder):
    tp = defaultdict(int)
    fp = defaultdict(int)
    fn = defaultdict(int)
    label_counts_total = defaultdict(int)
    model_counts_total = defaultdict(int)
    matching_files = defaultdict(list)

    for model_file in os.listdir(model_folder):
        if model_file.endswith('.txt'):
            nct_number = extract_nct_number(model_file)
            if nct_number:
                label_file_path = os.path.join(label_folder, f'{nct_number}.txt')
                model_file_path = os.path.join(model_folder, model_file)
                if os.path.exists(label_file_path):
                    label_text = read_file(label_file_path)
                    model_text = read_file(model_file_path)

                    label_operators = extract_operators(label_text)
                    model_operators = extract_operators(model_text)

                    label_counts = count_operators(label_operators)
                    model_counts = count_operators(model_operators)

                    for op in ['AND', 'OR', 'NOT']:
                        label_counts_total[op] += label_counts.get(op, 0)
                        model_counts_total[op] += model_counts.get(op, 0)
                        if label_counts.get(op, 0) == model_counts.get(op, 0):
                            tp[op] += 1
                            matching_files[op].append(nct_number)
                        else:
                            if label_counts.get(op, 0) > model_counts.get(op, 0):
                                fn[op] += 1
                            else:
                                fp[op] += 1

    for op in ['AND', 'OR', 'NOT']:
        precision, recall, f1, accuracy = calculate_metrics(tp[op], fp[op], fn[op])
        print(f"{op}:")
        print(f"  Precision: {precision:.3f}")
        print(f"  Recall: {recall:.3f}")
        print(f"  F1-score: {f1:.3f}")
        print(f"  Accuracy: {accuracy:.3f}")
        print(f"  Total in Label: {label_counts_total[op]}")
        print(f"  Total in Model: {model_counts_total[op]}")
        print(f"  Correctly Identified: {tp[op]}")
        #print(f"  Matching Files: {', '.join(matching_files[op])}\n")

label_folder = '../../input/lct_p1'
model_folder = 'model_output/Llama-3-70B-Instruct_4_shot/output'
process_files(label_folder, model_folder)


AND:
  Precision: 0.410
  Recall: 0.738
  F1-score: 0.527
  Accuracy: 0.358
  Total in Label: 819
  Total in Model: 1458
  Correctly Identified: 360
OR:
  Precision: 0.603
  Recall: 0.499
  F1-score: 0.546
  Accuracy: 0.376
  Total in Label: 4153
  Total in Model: 3383
  Correctly Identified: 378
NOT:
  Precision: 0.786
  Recall: 0.568
  F1-score: 0.660
  Accuracy: 0.492
  Total in Label: 948
  Total in Model: 445
  Correctly Identified: 495
